In [ ]:
# unzip file
!unzip bank+marketing .zip


unzip:  cannot find or open bank+marketing, bank+marketing.zip or bank+marketing.ZIP.


In [ ]:
!unzip bank.zip


Archive:  bank.zip
  inflating: bank-full.csv           
  inflating: bank-names.txt          
  inflating: bank.csv                


In [ ]:
import pandas as pd

# 1. Load dataset
df = pd.read_csv("bank-full.csv", sep=';')
df = df.replace('unknown', 'Not Specified')
df['converted'] = df['y'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# --- INSIGHT 1: Funnel & Drop-offs ---
total_traffic = len(df)
connected_leads = len(df[df['duration'] > 0])
engaged_prospects = len(df[df['duration'] > 120])
won_customers = df['converted'].sum()

funnel_df = pd.DataFrame({
    'Funnel Stage': ['Total Traffic', 'Connected Leads', 'Engaged Prospects', 'Converted Customers'],
    'Volume': [total_traffic, connected_leads, engaged_prospects, won_customers]
})
funnel_df['Conversion_Rate_%'] = (funnel_df['Volume'] / funnel_df['Volume'].shift(1) * 100).fillna(100).round(2)
funnel_df['Drop_off_Rate_%'] = (100 - funnel_df['Conversion_Rate_%']).round(2)

# --- INSIGHT 2: Channel Performance ---
channel_df = df.groupby('contact').agg(Total_Calls=('converted', 'count'), Total_Conversions=('converted', 'sum')).reset_index()
channel_df['Conversion_Rate_%'] = ((channel_df['Total_Conversions'] / channel_df['Total_Calls']) * 100).round(2)

# --- INSIGHT 3: Campaign Analysis (Fatigue) ---
campaign_df = df.groupby('campaign').agg(Total_Calls_Made=('converted', 'count'), Conversions=('converted', 'sum')).reset_index()
campaign_df['Conversion_Rate_%'] = ((campaign_df['Conversions'] / campaign_df['Total_Calls_Made']) * 100).round(2)
campaign_df = campaign_df.head(10)

# --- INSIGHT 4: Time Period (Month) ---
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
time_df = df.groupby('month').agg(Total_Contacts=('converted', 'count'), Conversions=('converted', 'sum')).reset_index()
time_df['Conversion_Rate_%'] = ((time_df['Conversions'] / time_df['Total_Contacts']) * 100).round(2)
time_df['month'] = pd.Categorical(time_df['month'], categories=month_order, ordered=True)
time_df = time_df.sort_values('month')

# --- INSIGHT 5: Customer Persona Profiles ---
persona_df = df.groupby(['job', 'education']).agg(Total_Audience=('converted', 'count'), Total_Conversions=('converted', 'sum')).reset_index()
persona_df['Conversion_Rate_%'] = ((persona_df['Total_Conversions'] / persona_df['Total_Audience']) * 100).round(2)
persona_df = persona_df.sort_values(by='Conversion_Rate_%', ascending=False).head(15)

# --- INSIGHT 6: Loan Status Risk Profiling ---
loan_df = df.groupby(['housing', 'loan']).agg(Total_Leads=('converted', 'count'), Total_Conversions=('converted', 'sum')).reset_index()
loan_df['Conversion_Rate_%'] = ((loan_df['Total_Conversions'] / loan_df['Total_Leads']) * 100).round(2)

# --- EXPORT TO NEW MASTER WORKBOOK ---
output_file = "Bank_Funnel_Advanced_Master.xlsx"
with pd.ExcelWriter(output_file) as writer:
    funnel_df.to_excel(writer, sheet_name="1_Funnel_Dropoffs", index=False)
    channel_df.to_excel(writer, sheet_name="2_Channel_Performance", index=False)
    campaign_df.to_excel(writer, sheet_name="3_Campaign_Analysis", index=False)
    time_df.to_excel(writer, sheet_name="4_Month_Analysis", index=False)
    persona_df.to_excel(writer, sheet_name="5_Customer_Personas", index=False)
    loan_df.to_excel(writer, sheet_name="6_Financial_Loans_Impact", index=False)

print(f"Success! Master data structured with 6 core insights. File '{output_file}' is ready.")


Success! Master data structured with 6 core insights. File 'Bank_Funnel_Advanced_Master.xlsx' is ready.
